In [15]:
from music21 import *

us = environment.UserSettings()

ruta_musescore = r"C:\Program Files\MuseScore 4\bin\MuseScore4.exe"


us['musicxmlPath'] = ruta_musescore
us['musescoreDirectPNGPath'] = ruta_musescore

# XML Conversor

In [ ]:
import subprocess
import sys
import os

musicxml_endpoint = input("intruduce nombre del archivo: ")

archivo_musicmxl = os.path.join("archivos", musicxml_endpoint)

def xml_to_abc(xml_path: str, unfold_repeats: bool = False) -> str:
    if not os.path.exists(xml_path):
        raise FileNotFoundError(f"No se encuentra el archivo: {xml_path}")

    script_path = os.path.join("xml2abc_177", "xml2abc.py")


    cmd = [sys.executable, script_path, "-n", "0", xml_path]
    if unfold_repeats:
        cmd.insert(3, "-u")

    result = subprocess.run(
        cmd, 
        capture_output=True, 
        text=True, 
        encoding="utf-8", 
        errors="replace"  
    )

    if result.returncode != 0:
        raise RuntimeError(f"Error de conversión: {result.stderr}")

    return result.stdout


abc_text = xml_to_abc(archivo_musicmxl)
print(abc_text)



X:1
T:Partitura sin título
C:Compositor / arreglista
%%score { 1 | 2 }
L:1/8
M:4/4
I:linebreak $
K:C
V:1 treble nm="Piano" snm="Pno."
V:2 bass 
L:1/4
V:1
 C2 C2 C2 E2 | A2 A2 A2 G2 | E2 E2 E2 E2 | B,2 B,2 B,2 A,2 :|$ CCCC CCEE | AAAA AAGG | EEEE EEEE | %7
 B,B,B,B, B,B,A,A, :|$ CCCC EEcB | AAAG GGGG | EEEE GFED | B,B,B,B, DDB,B, :| %12
V:2
 A,,4 | F,,4 | C,4 | G,,4 :|$ A,,4 | F,,4 | C,4 | G,,4 :|$ A,,4 | F,,4 | C,4 | G,,4 :| %12




# ABC Conversor

In [49]:
import subprocess
import sys
import os
import tempfile
import shutil

nombre_xml = input("Introduce el nombre del archivo de salida (ej. salida.musicxml): ")
if not nombre_xml.endswith((".xml", ".musicxml")):
    nombre_xml += ".musicxml"

def abc_to_xml_vree(abc_text: str, output_filename: str) -> str:
    """
    Convierte texto ABC a MusicXML usando abc2xml.py con el flag -o.
    """
    script_path = os.path.join("abc2xml", "abc2xml.py")
    if not os.path.exists(script_path):
        raise FileNotFoundError(f"No se encuentra el script en: {os.path.abspath(script_path)}")

    os.makedirs("output_xml", exist_ok=True)
    output_path = os.path.join("output_xml", output_filename)

  
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_abc_path = os.path.join(tmp_dir, "temp_score.abc")
        with open(tmp_abc_path, "w", encoding="utf-8") as f_abc:
            f_abc.write(abc_text)


        cmd = [sys.executable, script_path, "-o", tmp_dir, tmp_abc_path]
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace"
        )

        if result.returncode != 0:
            raise RuntimeError(f"Error en abc2xml (code {result.returncode}):\nSTDERR: {result.stderr}\nSTDOUT: {result.stdout}")

      
        generated_xml = os.path.join(tmp_dir, "temp_score.xml")
        
        if not os.path.exists(generated_xml) or os.path.getsize(generated_xml) == 0:
            raise RuntimeError(
                f"abc2xml no generó el XML esperado.\nSTDERR: {result.stderr}\nSTDOUT: {result.stdout}"
            )


        shutil.move(generated_xml, output_path)

    return output_path

xml_created = abc_to_xml_vree(abc_text, nombre_xml)
print(f"Archivo creado con éxito ({os.path.getsize(xml_created)} bytes): {xml_created}")

Archivo creado con éxito (27697 bytes): output_xml\salida.musicxml
